# 🫀 Heart Disease UCI — End-to-End MLOps Pipeline\n\n**Assignment:** MLOps Experimental Learning (S2-25_AMLCSZG523)  \n**Dataset:** Heart Disease UCI (Cleveland)  \n**Goal:** Build, track, and deploy a heart disease risk classifier on Google Cloud (GKE)\n\n---\n\n### Pipeline Overview\n1. Data Acquisition & EDA\n2. Feature Engineering & Model Development\n3. Experiment Tracking (MLflow)\n4. Model Packaging\n5. CI/CD & Containerization\n6. Production Deployment (GKE)

## 1. Setup & Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay, ConfusionMatrixDisplay
)

# Add project root to path
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_theme(style="whitegrid", palette="Set2")

print("✅ All libraries loaded successfully")

## 2. Data Acquisition & EDA

In [ ]:
COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope",
    "ca", "thal", "target"
]
FEATURE_COLS = COLUMNS[:-1]

DATA_PATH = "../data/processed.cleveland.data"
df_raw = pd.read_csv(DATA_PATH, header=None, names=COLUMNS, na_values="?")

print(f"Shape: {df_raw.shape}")
print(f"\nMissing values:\n{df_raw.isnull().sum()}")
df_raw.head(10)

In [ ]:
# Preprocess: binarize target
df = df_raw.copy()
df["target"] = (df["target"] > 0).astype(int)
df = df.dropna(subset=["target"])

print("Class distribution:")
print(df["target"].value_counts())
print(f"\nDisease prevalence: {df['target'].mean():.1%}")
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class balance
counts = df["target"].value_counts()
axes[0].bar(["No Disease", "Disease"], counts.values, color=["#2ecc71", "#e74c3c"], edgecolor="white", linewidth=1.5)
axes[0].set_title("Class Balance", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha="center", fontweight="bold")

# Age distribution by class
df[df["target"] == 0]["age"].hist(ax=axes[1], alpha=0.6, bins=20, color="#2ecc71", label="No Disease")
df[df["target"] == 1]["age"].hist(ax=axes[1], alpha=0.6, bins=20, color="#e74c3c", label="Disease")
axes[1].set_title("Age Distribution by Class", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.savefig("../models/class_balance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Feature histograms
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(FEATURE_COLS):
    df[col].hist(ax=axes[i], bins=20, edgecolor="white", color="#3498db", alpha=0.8)
    axes[i].set_title(col, fontweight="bold")
    axes[i].set_xlabel("")
# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Feature Distributions", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../models/feature_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

# Correlation heatmap
plt.figure(figsize=(12, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, square=True, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Heatmap", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("../models/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Feature Engineering & Model Training with MLflow

In [ ]:
X = df[FEATURE_COLS].values
y = df["target"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train disease rate: {y_train.mean():.2%} | Test disease rate: {y_test.mean():.2%}")

# MLflow setup
MLFLOW_URI = "../mlruns"
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("heart-disease-classification")

def build_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

def train_and_track(name, model, params):
    with mlflow.start_run(run_name=name):
        pipeline = build_pipeline(model)
        # 5-fold CV
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_auc = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="roc_auc")

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_proba),
            "cv_roc_auc_mean": cv_auc.mean(),
            "cv_roc_auc_std": cv_auc.std(),
        }
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(pipeline, artifact_path="model")
        print(f"\n{'='*50}")
        print(f"  {name}")
        print(f"{'='*50}")
        for k, v in metrics.items():
            print(f"  {k:25s}: {v:.4f}")
    return pipeline, metrics, y_proba

# Train Logistic Regression
lr_params = {"C": 1.0, "max_iter": 500, "solver": "lbfgs", "random_state": 42}
lr_pipeline, lr_metrics, lr_proba = train_and_track(
    "LogisticRegression",
    LogisticRegression(**lr_params),
    lr_params
)

# Train Random Forest
rf_params = {"n_estimators": 200, "max_depth": 8, "min_samples_split": 5, "random_state": 42}
rf_pipeline, rf_metrics, rf_proba = train_and_track(
    "RandomForest",
    RandomForestClassifier(**rf_params),
    rf_params
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC Curves
RocCurveDisplay.from_predictions(y_test, lr_proba, ax=axes[0], name="Logistic Regression")
RocCurveDisplay.from_predictions(y_test, rf_proba, ax=axes[0], name="Random Forest")
axes[0].set_title("ROC Curves", fontweight="bold")
axes[0].plot([0,1],[0,1],"k--", alpha=0.5)

# Confusion Matrix - LR
ConfusionMatrixDisplay.from_predictions(y_test, lr_pipeline.predict(X_test), ax=axes[1],
                                         display_labels=["No Disease", "Disease"],
                                         colorbar=False, cmap="Blues")
axes[1].set_title("Logistic Regression\nConfusion Matrix", fontweight="bold")

# Confusion Matrix - RF
ConfusionMatrixDisplay.from_predictions(y_test, rf_pipeline.predict(X_test), ax=axes[2],
                                         display_labels=["No Disease", "Disease"],
                                         colorbar=False, cmap="Oranges")
axes[2].set_title("Random Forest\nConfusion Matrix", fontweight="bold")

plt.tight_layout()
plt.savefig("../models/evaluation_plots.png", dpi=150, bbox_inches="tight")
plt.show()

# Feature Importance (RF)
importances = rf_pipeline.named_steps["model"].feature_importances_
sorted_idx = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 5))
plt.bar(range(len(FEATURE_COLS)), importances[sorted_idx], color="#3498db", edgecolor="white")
plt.xticks(range(len(FEATURE_COLS)), [FEATURE_COLS[i] for i in sorted_idx], rotation=45, ha="right")
plt.title("Random Forest — Feature Importances", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig("../models/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
## 4. Model Selection & Saving
os.makedirs("../models", exist_ok=True)

best_name = "RandomForest" if rf_metrics["roc_auc"] >= lr_metrics["roc_auc"] else "LogisticRegression"
best_pipeline = rf_pipeline if best_name == "RandomForest" else lr_pipeline
best_metrics = rf_metrics if best_name == "RandomForest" else lr_metrics

print(f"🏆 Best model: {best_name}")
print(f"   ROC-AUC: {best_metrics['roc_auc']:.4f}")
print(f"   Accuracy: {best_metrics['accuracy']:.4f}")

joblib.dump(best_pipeline, "../models/best_model.joblib")
joblib.dump(FEATURE_COLS, "../models/feature_cols.joblib")
print("\n✅ Model saved to ../models/best_model.joblib")

## 5. Local Inference Test\n\nVerify the saved model works correctly before containerization.

In [ ]:
loaded_model = joblib.load("../models/best_model.joblib")

# Sample patient (from row 0 of original data: Cleveland, no disease)
sample = np.array([[63.0, 1.0, 1.0, 145.0, 233.0, 1.0, 2.0, 150.0, 0.0, 2.3, 3.0, 0.0, 6.0]])
prediction = loaded_model.predict(sample)[0]
probability = loaded_model.predict_proba(sample)[0][1]

label = "❤️ Heart Disease Detected" if prediction == 1 else "✅ No Heart Disease"
print(f"Prediction: {label}")
print(f"Disease probability: {probability:.2%}")

## 6. Google Cloud Deployment (GKE)\n\n> Run the cells below **after** setting your Google Cloud project ID and ensuring `gcloud` and `kubectl` are installed and authenticated.\n\n### Prerequisites\n```bash\ngcloud auth login\ngcloud config set project YOUR_PROJECT_ID\ngcloud services enable container.googleapis.com containerregistry.googleapis.com\n```

In [ ]:
import subprocess, os

PROJECT_ID = os.getenv("GCP_PROJECT_ID", "YOUR_PROJECT_ID")   # ← Set your project ID here
IMAGE = f"gcr.io/{PROJECT_ID}/heart-disease-api"
CLUSTER = "heart-disease-cluster"
ZONE = "us-central1-a"

gke_commands = [
    # 1. Build & tag image
    f"docker build -t {IMAGE}:latest ../",
    # 2. Authenticate Docker with GCR
    "gcloud auth configure-docker --quiet",
    # 3. Push image to Google Container Registry
    f"docker push {IMAGE}:latest",
    # 4. Create GKE cluster (skip if exists)
    f"gcloud container clusters create {CLUSTER} --zone {ZONE} --num-nodes 2 --machine-type e2-standard-2 --enable-autoscaling --min-nodes 1 --max-nodes 4",
    # 5. Get credentials
    f"gcloud container clusters get-credentials {CLUSTER} --zone {ZONE}",
    # 6. Substitute image in manifest and apply
    f"sed 's|gcr.io/YOUR_GCR_PROJECT_ID/heart-disease-api:latest|{IMAGE}:latest|g' ../deployment/deployment.yaml | kubectl apply -f -",
    "kubectl apply -f ../deployment/service.yaml",
    "kubectl apply -f ../deployment/hpa.yaml",
    # 7. Wait for rollout
    "kubectl rollout status deployment/heart-disease-api --timeout=120s",
]

print("GKE Deployment commands:")
for i, cmd in enumerate(gke_commands, 1):
    print(f"  [{i}] {cmd}\n")
print("\n⚠️  Run these commands in sequence after setting PROJECT_ID above.")

## 7. Summary & Next Steps

### Model Comparison
| Metric | Logistic Regression | Random Forest |
|--------|-------------------|---------------|
| Accuracy | ~0.87 | ~0.89 |
| Precision | ~0.81 | ~0.84 |
| Recall | ~0.93 | ~0.93 |
| ROC-AUC | ~0.95 | ~0.95 |
| CV ROC-AUC | ~0.90 | ~0.88 |

### Deployment Checklist
- [x] Data cleaned & preprocessed
- [x] Two models trained (Logistic Regression + Random Forest)
- [x] MLflow experiment tracking
- [x] Model saved as joblib artifact
- [x] FastAPI serving app with /predict, /health, /metrics
- [x] Unit tests (pytest)
- [x] Dockerfile & .dockerignore
- [x] CI/CD pipeline (GitHub Actions)
- [x] GKE deployment manifests (Deployment + Service + HPA)
- [x] Prometheus + Grafana monitoring
- [x] Documentation (README.md)

### Deployment Command
```bash
export GCP_PROJECT_ID=your-project-id
./deploy.sh
```